# Random Forest
## DS-3001: Foundations of Machine Learning

Content adapted from Terence Johnson (UVA)

### Loading Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os # For changing directory

# To mount your google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_to_DS_3001_folder = '/content/drive/MyDrive/DS-3001/03_Tuning_Testing_Evaluating'
# path_to_DS_3001_folder = ''

# Update the path to your folder for the class
# Where you stored the data from the previous noteboook
os.chdir(path_to_DS_3001_folder)

# Random Forests

## Random Forests

- Previously, we looked at how we can use bootstrapping to construct ensemble models that on average perform better than individual predictors.

- Next, we'll look at **Random Forest**, which expands this idea to further tune the performance of many decision trees aggregated together.

#### How do we make random forest models?

- A **random forest** is constructed by:

  1. Select a bootstrap sample from the data
  2. Fit a decision tree as normal, but at each split in the tree, **select a random subset of features/predictors/variables to consider** (random subspace method)
  3. Repeat the above steps a large number of times

- To make a prediction, use all the trees in the forest
  1. For **regression**, average the individual results
  2. For **classification**, take the most predicted category

- This can be pretty computationally intensive

### Random Forests with `sklearn`

- The random forest functions in sklearn are:
  1. For regression: [sklearn.ensemble.RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
  2. For classificaiton: [sklearn.ensemble.RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

- The key inputs to be aware of are:
  `n_estimators`: The number of trees to fit for the random forest.
  `max_depth`: The max depth for each individual tree.
  `min_samples_leaf`: The minimum number of samples that can be in a leaf.
  `random_state`: The random seed to make your model reproducible.


In [ ]:
from sklearn.model_selection import train_test_split

# Load in our data set again from the Ensembles notebook
house_df = pd.read_csv('./data/pierce_county_house_sales.csv')

# Create an age variable
house_df['age'] = max(house_df['year_built']) - house_df['year_built']

# Select variables of interest to look at
vars = [
    "attached_garage_square_feet", "attic_finished_square_feet",
    "basement_square_feet", "bathrooms",
    "bedrooms", "detached_garage_square_feet",
    "fireplaces", "house_square_feet",
    "stories", "age", "sale_price"
]

# Subset to get the variables of interest
house_df = house_df.loc[:,vars]

# Subset our data into features and outcomes
y = house_df['sale_price']
X = house_df.drop('sale_price', axis = 1)

# Split our data into train and test sets
random_state = 3001
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = random_state
)

In [ ]:
# The import to get the random forest regression model function
from sklearn.ensemble import RandomForestRegressor

# Fit a Random Forest regression model


In [ ]:
# Scatterplot of predicted price vs. actual (Our residual)
sns.scatterplot(
    y= y_test,
    x= y_hat_rf,
    alpha=.25
)
plt.title('Random Forest Predictions vs True Outcome Price')
plt.xlabel('Test Outcome Price')
plt.ylabel('Random Forest Predictions')

# Finding the min and max of the x and y-axes to set up the 45 degree line
x_min = min(y_test)
x_max = max(y_test)
y_min = min(y_hat_rf)
y_max = max(y_hat_rf)

# Plotting the 45 degree line
plt.plot(
    [x_min, x_max],
    [y_min, y_max],
    color='red',
    linestyle = '--'
)

plt.show()

In [ ]:
# How are our residuals distributed?

# Calculate the residual
residual = y_test - y_hat_rf

# Plot the residual as a KDE plot
sns.kdeplot(x=residual)
plt.title('Random Forest Residual Distribution')
plt.xlabel(r'Residual ($y - \hat{y}$)')
plt.show()

**How does our random forest model compare to the results from when we randomly ensembled 100 trees? Previously we had an $R^2$ of ~0.36**

In [ ]:
# Compute the R^2 of the model on the test data
# How does it compare to the ensemble model we fit before?


**Question:** How does it compare to the ensemble model we fit before?

**Answer:**

# Feature Importance

### Model Explainability & Interpretability

- We're reaching a point where the models become hard to **explain** to people and hard for us to **interpret**

- **What makes a model explainable:**
  - A model is **explainable** if the mapping from inputs to predictions can be understood by human intelligence (how does the model use inputs/data to create predictions?)

- **What makes a model interpretable:**
  - A model is **interpretable** if any prediction can be rationalized from the data and model by human intelligence (why is the model making this particular prediction?)

- The random forest, along with neural networks, is one model that probably isn't explainable or interpretable in these terms, since it aggregates many models trained on randomly generated subsets of data.

###  How do we Evaluate Variable Importance for Random Forest?

- **Often times we want some interpretable output**. For example, a regression coefficient table.

- Since the random forest is an average of hundreds of randomly constructed objects, interpretability is more challenging than something like linear regression.

- For random forest models, people report **variable importance**.
  - This is a way of suggesting **which variables ended up being the most useful for the process of constructing the trees** in the forest.

- **How do you interpret variable importance?**
  - It is the average amount by which **Gini impurity/Entropy/MSE** is reduced when this variable is used across all trees in the ensemble. This value is normalized so they sum to 1 across features.

- **How does this differ from regression coefficients?**
  - The feature importance is measured in impurity (impact on error), **not** the impact on the predictions.
  - In contrast, a regression coefficient describes how the prediction changes according to the feature.

- **Accessing feature importance in sklearn:**
  - You can access the feature importance of your random forest object (`rf`) as `rf.feature_importances_`.

In [ ]:
# Isolate the feature importances as a Pandas Series
feature_importances_series = pd.Series(
    rf.feature_importances_, # How we are accessing the feature importancesc
    index = X_train.columns
)

# Sort the values
feature_importances_series = feature_importances_series.sort_values()

# Plot the results as a bar graph
feature_importances_series.plot(kind='barh')

# Add labels for the title and x-axis
plt.title('Feature Importance')
plt.xlabel('Normalized Mean decrease in impurity (MSE)')
plt.show()

# Random Forests For Classification Tasks

- We've focused on regression, but you can obviously do this with **classification** as well.

- Instead of predicting a numeric value, **each weak learner (tree) predicts a categorical value**

- The weak learners "vote" over the winning prediction: **The label with the most votes wins** (or, you can report the frequency of the votes)

### Data Set for Classification

- For this tasks, we'll use the `corporate_ratings.csv` file.

- We'll be predicting the bond rating (`Rating`) based on information about the company.

- The **observations/rows** in this data set are the company at different dates and from different rating agencies.

- Here is a [wikipedia article about bond ratings](https://en.wikipedia.org/wiki/Bond_credit_rating) for more background information if you're interested.

- The best rating is **AAA** and the worst is **D**.

In [ ]:
# Load in the data set
corporate_df = pd.read_csv('./data/corporate_ratings.csv')

# Look at the shape and first 5 rows of the data set
print(corporate_df.shape)
corporate_df.head()

In [ ]:
# We can look at the distribution of the ratings

# First look at the raw values counts of the ratings
print(corporate_df['Rating'].value_counts())

# Look at the histogram for the distribution
corporate_df['Rating'].sort_values().hist(bins=20)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# Isolate our outcome variable
y = corporate_df['Rating']

# Drop some columns from our design matrix that are not likely informative
X = corporate_df.drop(
    [
        'Rating',
        'Date',
        'Name',
        'Symbol',
        'Rating Agency Name',
        'Sector'
    ],
    axis=1
)

# Feature engineering for the sector and agency names
sector = pd.get_dummies(corporate_df['Sector'], dtype=int)
agency = pd.get_dummies(corporate_df['Rating Agency Name'], dtype=int)

# Concatinating back to a single design matrix
X = pd.concat(
    [
        X, # Orginal design matrix
        agency, # The dummy encoding of agency
        sector # The dummy encoding of sector
    ],
    axis=1
)

# Compute a train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size =.2,
    random_state=100
)

In [ ]:
# The function necessary for random forest classification
from sklearn.ensemble import RandomForestClassifier

# Defining our model instance
rf = RandomForestClassifier(
    n_estimators = 100, # Number of trees to have in the random forest
    min_samples_leaf = 35, # The minimum number of samples that can be in a leaf
    max_depth = 10 # The maximum depth the trees can have
)

# Fitting the model to our training data
rf.fit(X_train, y_train)

# Getting our predictions on the test data set
y_hat = rf.predict(X_test)

# Computing the confusion matrix
print(pd.crosstab(y_test,y_hat))

# Calculating the accuracy from the confusion matrix
acc = np.trace(pd.crosstab(y_test,y_hat)) / len(y_test)
print(acc)

**Question:** Look at the predictions.
  - Are we predciting all classes?
  - What's the most common predicted class?
  - Why might this be?

In [ ]:
# Answer Here


**Computing the variable importance for our result:**

In [ ]:
# Variable importance plot for the classification task
feature_importances = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
)

# Sorting the values
feature_importances = feature_importances.sort_values()

# Plotting the feature importances
feature_importances.plot(kind='barh')
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Mean decrease in impurity')
plt.tight_layout()
plt.show()

**This model performs pretty poorly. We could try to further fine tune it using our hyperparameters.**

In [ ]:
# Hyperparameter tune for min_samples_leaf
import tqdm as tqdm

min_samples_per_leaf = np.arange(1, 100, 10)

train_accuracies = []
test_accuracies = []

for min_sample_per_leaf in tqdm.tqdm(min_samples_per_leaf):
  # Defining our model instance
  rf = RandomForestClassifier(
      n_estimators = 100, # Number of trees to have in the random forest
      min_samples_leaf = min_sample_per_leaf, # The minimum number of samples that can be in a leaf
      max_depth = 10 # The maximum depth the trees can have
  )

  # Fitting the model to our training data
  rf.fit(X_train, y_train)

  # Getting our predictions on the test data set
  y_hat_train = rf.predict(X_train)
  y_hat_test = rf.predict(X_test)

  # Calculating the accuracy from the confusion matrix
  train_acc = np.trace(pd.crosstab(y_train, y_hat_train)) / len(y_train)
  test_acc = np.trace(pd.crosstab(y_test, y_hat_test)) / len(y_test)

  # Save the results
  train_accuracies.append(train_acc)
  test_accuracies.append(test_acc)

In [ ]:
# Plot the results
plt.plot(min_samples_per_leaf, train_accuracies, label = 'Train', marker = 'o')
plt.plot(min_samples_per_leaf, test_accuracies, label = 'Test', marker = 'o')
plt.xlabel('Min Number of Samples Per Leaf')
plt.ylabel('Accuarcy')
plt.title('Random Forest Hyperparameter Search:\nMin Samples Per Leaf')
plt.legend()
plt.show()

In [ ]:
# Hyperparameter tune for min_samples_leaf
import tqdm as tqdm

max_depths = np.arange(1, 100, 10)

train_accuracies = []
test_accuracies = []

for max_depth in tqdm.tqdm(max_depths):
  # Defining our model instance
  rf = RandomForestClassifier(
      n_estimators = 100, # Number of trees to have in the random forest
      min_samples_leaf = 1, # The minimum number of samples that can be in a leaf
      max_depth = max_depth # The maximum depth the trees can have
  )

  # Fitting the model to our training data
  rf.fit(X_train, y_train)

  # Getting our predictions on the test data set
  y_hat_train = rf.predict(X_train)
  y_hat_test = rf.predict(X_test)

  # Calculating the accuracy from the confusion matrix
  train_acc = np.trace(pd.crosstab(y_train, y_hat_train)) / len(y_train)
  test_acc = np.trace(pd.crosstab(y_test, y_hat_test)) / len(y_test)

  # Save the results
  train_accuracies.append(train_acc)
  test_accuracies.append(test_acc)

In [ ]:
# Plot the results
plt.plot(min_samples_per_leaf, train_accuracies, label = 'Train', marker = 'o')
plt.plot(min_samples_per_leaf, test_accuracies, label = 'Test', marker = 'o')
plt.xlabel('Max Depth per Tree')
plt.ylabel('Accuarcy')
plt.title('Random Forest Hyperparameter Search:\nMax Depth')
plt.legend()
plt.show()

# Conclusion

### Benefits of  Ensembles

- **Ensembles improve performance through bootstrap aggregation:**
  - Ensemble learning can improve the performance of any kind of base learner (e.g. decision trees, linear models) through "**bootstrap aggregation**" (model averaging)

- These ideas can also motivate more complex uses of randomization to improve performance:
  - The intuition is to **make the ensemble members as uncorrelated** as possible, but still trained on the same data, so their different predictions end up complementing one another.

- The Random Forest is a complex and computationally intensive algorithm, but it **resolves the fragility problem for decision-trees by using bootstrap aggregation and random feature selection**

- This is the first payoff of adopting a more probabilistic perspective on the data

### Extending Random Forest: Gradient Boosting

- The other ensemble learning method that we won't talk about is called *Gradient Boosting*

- A problem with decision trees is how much information they fail to use:
  - Binary, greedy splits can end up using the same variable or subset of variables extensively, without touching others

- Reframing the learning objective based on residuals:
  - **Take the residual from one decision tree**, and use it as the target/outcome variable for a second set of models

- This is danger zone overfitting territory, but can be done in thoughtful and powerful ways, like extreme gradient boosting (e.g. `XGBoost`, `AdaBoost`)